# **Dati dei Comuni Italiani**

In [ ]:
!pip install pandas sqlalchemy psycopg2-binary

Eseguo il mount della cartella di Google Drive ove sono presenti i dataset da caricare ed imposto il flag per indicare se il salvataggio del dataset deve essere fatto su database oppure su file excel

In [ ]:
from google.colab import drive
# Smonta il drive per azzerare la cache di sessione
drive.flush_and_unmount()
# Eseguo il mount del drive Google
drive.mount('/content/drive')
# Flag per indicare se il salvataggio del dataset di produzione deve essere
# eseguito su database oppure su file excel
FLAG_SALVATAGGIO_DB = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Creo la connessione al database

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy import text
from google.colab import userdata

# Configurazione database
DB_USER = "postgres.luhxmgsxvbkfuylgkthr"
# Inserire nei secret di colab la password per l'accesso al database
DB_PASSWORD = userdata.get("SUPABASE_PASSWORD")
DB_HOST = "aws-0-eu-west-1.pooler.supabase.com"
DB_PORT = "5432"
DB_NAME = "postgres"

# Connessione a Postgres tramite SQLAlchemy
DATABASE_URL = (
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Creo l'engine tramite la funzione create_engine di SQLAlchemy
engine = create_engine(DATABASE_URL)

Carico i dataset nei dataframe pandas

In [ ]:
import os as os
import pandas as pd

# Nomi delle tabelle di produzione
TAB_COMUNI_COSTI_PROD = "comuni_costi"
TAB_COMUNI_RIFIUTI_PROD = "comuni_rifiuti"
TAB_COMUNI_PUNTI_RACCOLTA_PROD = "comuni_punti_raccolta"

# Configurazione percorso in Colab ove sono presenti i file csv da caricare
CARTELLA_COMUNI = "/content/drive/MyDrive/Ricicla(MI)/Citta_di_Milano/dataset/"
CARTELLA_OUTPUT = "/content/drive/MyDrive/Ricicla(MI)/output/"

# Dataset
FILE_INDICATORE_COSTI = "indicatori_costi_milano.csv"
FILE_ANDAMENTO_RIFIUTI = "andamento_rifiuti_milano.csv"
FILE_PUNTI_RACCOLTA = "punti_raccolta_milano.csv"

# File di output
FILE_INDICATORE_COSTI_OUT = "indicatori_costi.xlsx"
FILE_ANDAMENTO_RIFIUTI_OUT = "andamento_rifiuti.xlsx"
FILE_PUNTI_RACCOLTA_OUT = "punti_raccolta.xlsx"

# Verifica che la cartella esista e carico i files csv in caso di esito positivo
if not os.path.exists(CARTELLA_COMUNI):
    print(
        f"ERRORE: La cartella '{CARTELLA_COMUNI}' non esiste. Assicurati di avere i permessi nececessari per l'accesso a Google Drive."
    )
else:
  print(f"-> Leggo il file '{FILE_INDICATORE_COSTI}'")
  df_costi = pd.read_csv(CARTELLA_COMUNI + FILE_INDICATORE_COSTI, sep=",")
  print(f"-> Leggo il file '{FILE_ANDAMENTO_RIFIUTI}'")
  df_rifiuti = pd.read_csv(CARTELLA_COMUNI + FILE_ANDAMENTO_RIFIUTI, sep=",")
  print(f"-> Leggo il file '{FILE_PUNTI_RACCOLTA}'")
  df_punti = pd.read_csv(CARTELLA_COMUNI + FILE_PUNTI_RACCOLTA, sep=",")


-> Leggo il file 'indicatori_costi_milano.csv'
-> Leggo il file 'andamento_rifiuti_milano.csv'
-> Leggo il file 'punti_raccolta_milano.csv'


Correggo eventuali errori nel dataframe dei costi:
* formatto il codice_istat a 6 cifre
* su tutte le colonne di tipo float sostituisco i valori nan con la mediana dei valori sulla stessa colonna (i file caricati si riferiscono solo al comune di milano)

In [ ]:
import numpy as np

# Sistemo le colonne del dataframe
try:
  df_costi["codice_istat"] = df_costi["codice_istat"].astype(str).str.zfill(6)

  colonne_float = df_costi.select_dtypes(include=['float']).columns
  for col in colonne_float:
    # Sostituisci 0 con NaN prima di calcolare la mediana
    df_costi.loc[df_costi[col] == 0, col] = np.nan

    # Calcola la mediana (restituisce un singolo numero float)
    mediana_colonna = df_costi[col].median()

    # Verifica se la mediana è un numero valido prima di usarla per riempire i NaN
    if pd.notna(mediana_colonna):
      df_costi[col] = df_costi[col].fillna(mediana_colonna)

except Exception as e:
  print(f"Eccezione: {e}")

Eseguiamo un controllo di data quality per verificare se ci sono valori vuoti o mancanti nel dataframe

In [ ]:
#data quality
import numpy as np
# 1. Standardizza i vuoti: trasforma spazi vuoti e stringhe vuote in NaN
df_controllo = df_costi.replace([r'^\s*$', 'None', 'NaN'], np.nan, regex=True)

# 2. Verifica se esiste ALMENO un valore nullo in tutto il DataFrame
if df_controllo.isna().any().any():
    print("⚠️ ALERT: Ci sono valori vuoti o mancanti all'interno del DataFrame!")

    # OPZIONALE: Mostra quali colonne contengono i vuoti e quanti sono
    conteggio_vuoti = df_controllo.isna().sum()
    print("\nDettaglio dei vuoti per colonna:")
    print(conteggio_vuoti[conteggio_vuoti > 0])
else:
    print("✅ Ottimo! Il DataFrame è completamente pulito e non ha valori vuoti.")


✅ Ottimo! Il DataFrame è completamente pulito e non ha valori vuoti.


Scrivo su database il dataset

In [ ]:
# Creo la nuova tabella di produzione sovrascrivendo eventualmente quella già
# esistente
if FLAG_SALVATAGGIO_DB:
  with engine.begin() as connection:
    df_costi.to_sql(
        name=TAB_COMUNI_COSTI_PROD,
        con=connection,  # Passa la connessione attiva, NON l'engine
        if_exists='replace',
        index=False,
        chunksize=10000,
        method='multi'
  )
else:
  df_costi.to_excel(CARTELLA_OUTPUT + FILE_INDICATORE_COSTI_OUT, index=False)
  print(f"DataFrame salvato nel file Excel {FILE_INDICATORE_COSTI_OUT}")
# Metto a null il dataframe
del df_costi

DataFrame salvato nel file Excel indicatori_costi.xlsx


/tmp/ipykernel_2495/866680636.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_costi[:] = np.nan
/tmp/ipykernel_2495/866680636.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_costi[:] = np.nan


In [ ]:
import numpy as np

# Sistemo le colonne del dataframe
try:
  df_rifiuti["codice_istat"] = df_rifiuti["codice_istat"].astype(str).str.zfill(6)

  colonne_float = df_rifiuti.select_dtypes(include=['float']).columns
  for col in colonne_float:
    # Sostituisci 0 con NaN prima di calcolare la mediana
    df_rifiuti.loc[df_rifiuti[col] == 0, col] = np.nan

    # Calcola la mediana (restituisce un singolo numero float)
    mediana_colonna = df_rifiuti[col].median()

    # Verifica se la mediana è un numero valido prima di usarla per riempire i NaN
    if pd.notna(mediana_colonna):
      df_rifiuti[col] = df_rifiuti[col].fillna(mediana_colonna)

except Exception as e:
  print(f"Eccezione: {e}")

In [ ]:
#data quality
import numpy as np
# 1. Standardizza i vuoti: trasforma spazi vuoti e stringhe vuote in NaN
df_controllo = df_rifiuti.replace([r'^\s*$', 'None', 'NaN'], np.nan, regex=True)

# 2. Verifica se esiste ALMENO un valore nullo in tutto il DataFrame
if df_controllo.isna().any().any():
    print("⚠️ ALERT: Ci sono valori vuoti o mancanti all'interno del DataFrame!")

    # OPZIONALE: Mostra quali colonne contengono i vuoti e quanti sono
    conteggio_vuoti = df_controllo.isna().sum()
    print("\nDettaglio dei vuoti per colonna:")
    print(conteggio_vuoti[conteggio_vuoti > 0])
else:
    print("✅ Ottimo! Il DataFrame è completamente pulito e non ha valori vuoti.")


✅ Ottimo! Il DataFrame è completamente pulito e non ha valori vuoti.


In [ ]:
# Creo la nuova tabella di produzione sovrascrivendo eventualmente quella già
# esistente
if FLAG_SALVATAGGIO_DB:
  with engine.begin() as connection:
    df_rifiuti.to_sql(
        name=TAB_COMUNI_RIFIUTI_PROD,
        con=connection,  # Passa la connessione attiva, NON l'engine
        if_exists='replace',
        index=False,
        chunksize=10000,
        method='multi'
   )
else:
  df_rifiuti.to_excel(CARTELLA_OUTPUT + FILE_ANDAMENTO_RIFIUTI_OUT, index=False)
  print(f"DataFrame salvato nel file Excel {FILE_ANDAMENTO_RIFIUTI_OUT}")
# Metto a null il dataframe
del df_rifiuti

DataFrame salvato nel file Excel andamento_rifiuti.xlsx


/tmp/ipykernel_2495/917219405.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_rifiuti[:] = np.nan


In [ ]:
# Sistemo le colonne del dataframe
try:
  df_punti["codice_istat"] = df_punti["codice_istat"].astype(str).str.zfill(6)
  id_stringa = df_punti["id_punto"].astype(str)
  riga_vuota = df_punti["codice_originale"].isna() | (df_punti["codice_originale"].astype(str).str.strip() == "") | (df_punti["codice_originale"] == "NaN")
  df_punti.loc[riga_vuota, "codice_originale"] = "ECOISOLA_" + id_stringa
except Exception as e:
  print(f"Eccezione: {e}")

In [ ]:
#data quality
import numpy as np
# 1. Standardizza i vuoti: trasforma spazi vuoti e stringhe vuote in NaN
df_controllo = df_punti.replace([r'^\s*$', 'None', 'NaN'], np.nan, regex=True)

# 2. Verifica se esiste ALMENO un valore nullo in tutto il DataFrame
if df_controllo.isna().any().any():
    print("⚠️ ALERT: Ci sono valori vuoti o mancanti all'interno del DataFrame!")

    # OPZIONALE: Mostra quali colonne contengono i vuoti e quanti sono
    conteggio_vuoti = df_controllo.isna().sum()
    print("\nDettaglio dei vuoti per colonna:")
    print(conteggio_vuoti[conteggio_vuoti > 0])
else:
    print("✅ Ottimo! Il DataFrame è completamente pulito e non ha valori vuoti.")


✅ Ottimo! Il DataFrame è completamente pulito e non ha valori vuoti.


In [ ]:
# Creo la nuova tabella di produzione sovrascrivendo eventualmente quella già
# esistente
if FLAG_SALVATAGGIO_DB:
  with engine.begin() as connection:
    df_punti.to_sql(
        name=TAB_COMUNI_PUNTI_RACCOLTA_PROD,
        con=connection,  # Passa la connessione attiva, NON l'engine
        if_exists='replace',
        index=False,
        chunksize=10000,
        method='multi'
  )
else:
  df_punti.to_excel(CARTELLA_OUTPUT + FILE_PUNTI_RACCOLTA_OUT, index=False)
  print(f"DataFrame salvato nel file Excel {FILE_PUNTI_RACCOLTA_OUT}")
# Metto a null il dataframe
del df_punti

DataFrame salvato nel file Excel punti_raccolta.xlsx


/tmp/ipykernel_2495/367446816.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_punti[:] = np.nan
/tmp/ipykernel_2495/367446816.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_punti[:] = np.nan
/tmp/ipykernel_2495/367446816.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_punti[:] = np.nan


In [ ]:
# Chiude la connessione e rilascia le risorse del pool
engine.dispose()